# Tarea 4 — Componente Práctico
**Curso:** Lenguajes Interpretados (202037696)  
**Estudiante:** Javier Orlando Santanilla Moncada  
**Escenario 1:** Procesamiento de alertas por sobrecorriente (subestación eléctrica)

Este cuaderno mide el tiempo de CPU del script industrial provisto en la guía, audita sus faltas al estándar PEP 8, lo refactoriza y vuelve a medirlo con **exactamente los mismos parámetros**, verificando que la lógica de protección (umbral de 25.0 A) no cambie.

## Paso 2 — Diagnóstico temporal del código original
El bloque siguiente se copia **exactamente** como aparece en la guía. `%%writefile` lo guarda en un archivo para auditarlo después con `flake8`, y `%run` lo carga en memoria.

In [ ]:
%%writefile escenario1_original.py
# ESCENARIO 1: Copie exactamente este bloque en Google Colab
HISTORIAL_DE_LECTURAS_DE_CORRIENTE_DE_LA_SUBESTACION_PRINCIPAL_A = [1.5, 2.0, 12.8, 4.5, 30.2, 1.1] * 5000
def BUSCAR_SI_EXISTE_UN_VALOR_CRITICO_EN_EL_SISTEMA_DE_RELEVADORES(VALOR):
    # Ineficiencia masiva: recorre millones de datos de forma lineal una y otra vez
    EXISTE = False
    for LECTURA in HISTORIAL_DE_LECTURAS_DE_CORRIENTE_DE_LA_SUBESTACION_PRINCIPAL_A:
        if LECTURA == VALOR: EXISTE = True
    if EXISTE == True and VALOR > 25.0: print("ALERTA DE SOBRECORRIENTE DETECTADA EN LA SUBESTACIÓN: EL VALOR SUPERA LOS LÍMITES")
    return EXISTE

Writing escenario1_original.py


In [ ]:
%run escenario1_original.py

### Parámetros de medición (idénticos en el Paso 4)
- **Lote de lecturas simuladas del transductor:** `[1.5, 12.8, 30.2, 7.7]` A — dos lecturas normales registradas, una crítica (30.2 A > 25.0 A) y una no registrada.
- **Medición:** `%timeit -n 100 -r 7` → 7 repeticiones de 100 ejecuciones del lote.
- Las alertas se redirigen a un búfer durante la medición para que la escritura en pantalla no contamine el tiempo de CPU; la misma condición se aplica a ambas versiones.

In [ ]:
import contextlib
import io

lecturas_simuladas = [1.5, 12.8, 30.2, 7.7]


def procesar_lote(funcion, lecturas):
    """Procesa un lote de lecturas sin imprimir las alertas en pantalla."""
    with contextlib.redirect_stdout(io.StringIO()):
        for lectura in lecturas:
            funcion(lectura)

In [ ]:
t_original = %timeit -o -n 100 -r 7 procesar_lote(BUSCAR_SI_EXISTE_UN_VALOR_CRITICO_EN_EL_SISTEMA_DE_RELEVADORES, lecturas_simuladas)
print(f"Código original: {t_original.average * 1e6:,.2f} µs por lote de {len(lecturas_simuladas)} lecturas")

3.83 ms ± 69.4 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Código original: 3,831.97 µs por lote de 4 lecturas


## Paso 3 — Auditoría PEP 8 y refactorización
Se usa `flake8` con el complemento `pep8-naming`, que verifica automáticamente el estándar PEP 8 (incluidas las reglas de nomenclatura). Límite de línea: 79 caracteres.

In [ ]:
!pip install -q flake8 pep8-naming
!flake8 --max-line-length=79 escenario1_original.py

escenario1_original.py:2:80: E501 line too long (106 > 79 characters)
escenario1_original.py:3:1: E302 expected 2 blank lines, found 0
escenario1_original.py:3:6: N802 function name 'BUSCAR_SI_EXISTE_UN_VALOR_CRITICO_EN_EL_SISTEMA_DE_RELEVADORES' should be lowercase
escenario1_original.py:3:69: N803 argument name 'VALOR' should be lowercase
escenario1_original.py:4:80: E501 line too long (83 > 79 characters)
escenario1_original.py:5:6: N806 variable 'EXISTE' in function should be lowercase
escenario1_original.py:6:10: N806 variable 'LECTURA' in function should be lowercase
escenario1_original.py:6:80: E501 line too long (84 > 79 characters)
escenario1_original.py:7:28: E701 multiple statements on one line (colon)
escenario1_original.py:7:31: N806 variable 'EXISTE' in function should be lowercase
escenario1_original.py:8:15: E712 comparison to True should be 'if cond is True:' or 'if cond:'
escenario1_original.py:8:39: E701 multiple statements on one line (colon)
escenario1_original.py:

**Ineficiencia algorítmica:** cada consulta recorre las 30 000 posiciones de la lista (búsqueda lineal, O(N)) y no se detiene aunque ya haya encontrado el valor. La lista solo contiene 6 valores distintos, así que un conjunto (`set`) construido una vez permite responder cada consulta en tiempo constante, O(1), sin cambiar el resultado.

### Código refactorizado

In [ ]:
%%writefile escenario1_optimizado.py
# AUTOR: JAVIER ORLANDO SANTANILLA MONCADA
# Lenguajes Interpretados: 202037696_65
# ESCENARIO 1: Procesamiento de alertas por sobrecorriente (optimizado)
UMBRAL_SOBRECORRIENTE_A = 25.0

historial_corriente_a = [1.5, 2.0, 12.8, 4.5, 30.2, 1.1] * 5000

# Índice de acceso directo: se construye una sola vez a partir del
# historial y permite verificar cada lectura en tiempo constante, O(1).
lecturas_registradas = set(historial_corriente_a)


def buscar_valor_critico(valor):
    """Indica si la lectura existe en el historial y alerta si es crítica."""
    existe = valor in lecturas_registradas
    if existe and valor > UMBRAL_SOBRECORRIENTE_A:
        print("ALERTA DE SOBRECORRIENTE DETECTADA EN LA SUBESTACIÓN: "
              "EL VALOR SUPERA LOS LÍMITES")
    return existe

Writing escenario1_optimizado.py


In [ ]:
%run escenario1_optimizado.py
!flake8 --max-line-length=79 escenario1_optimizado.py && echo "0 violaciones PEP 8"

0 violaciones PEP 8


## Paso 4 — Contra-verificación
### 4.1 Equivalencia funcional
Ambas versiones deben devolver lo mismo y emitir la misma alerta para cualquier lectura, incluidos los casos límite (25.0 A exactos y lecturas no registradas).

In [ ]:
pruebas = [1.5, 2.0, 12.8, 4.5, 30.2, 1.1, 7.7, 25.0, 30.3, 0.0]

for valor in pruebas:
    salida_original, salida_optimizada = io.StringIO(), io.StringIO()
    with contextlib.redirect_stdout(salida_original):
        r_original = BUSCAR_SI_EXISTE_UN_VALOR_CRITICO_EN_EL_SISTEMA_DE_RELEVADORES(valor)
    with contextlib.redirect_stdout(salida_optimizada):
        r_optimizado = buscar_valor_critico(valor)
    assert r_original == r_optimizado, valor
    assert salida_original.getvalue() == salida_optimizada.getvalue(), valor

print(f"Equivalencia verificada en {len(pruebas)} lecturas: mismo resultado "
      "y misma alerta.")

Equivalencia verificada en 10 lecturas: mismo resultado y misma alerta.


### 4.2 Alertas técnicas emitidas por el código optimizado

In [ ]:
for lectura in lecturas_simuladas:
    registrada = buscar_valor_critico(lectura)
    print(f"Lectura {lectura:5.1f} A -> registrada en el historial: {registrada}")

Lectura   1.5 A -> registrada en el historial: True
Lectura  12.8 A -> registrada en el historial: True
ALERTA DE SOBRECORRIENTE DETECTADA EN LA SUBESTACIÓN: EL VALOR SUPERA LOS LÍMITES
Lectura  30.2 A -> registrada en el historial: True
Lectura   7.7 A -> registrada en el historial: False


### 4.3 Nueva marca de tiempo (mismos parámetros del Paso 2)

In [ ]:
t_optimizado = %timeit -o -n 100 -r 7 procesar_lote(buscar_valor_critico, lecturas_simuladas)
print(f"Código optimizado: {t_optimizado.average * 1e6:,.2f} µs por lote de {len(lecturas_simuladas)} lecturas")

5.69 µs ± 2.28 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Código optimizado: 5.69 µs por lote de 4 lecturas


### 4.4 Resumen de rendimiento

In [ ]:
import sys

us_original = t_original.average * 1e6
us_optimizado = t_optimizado.average * 1e6
reduccion = (1 - us_optimizado / us_original) * 100
aceleracion = us_original / us_optimizado

print(f"Tiempo original   : {us_original:12,.2f} µs por lote")
print(f"Tiempo optimizado : {us_optimizado:12,.2f} µs por lote")
print(f"Reducción del tiempo de CPU: {reduccion:.2f} %")
print(f"Aceleración: {aceleracion:,.0f} veces más rápido")

Tiempo original   :     3,831.97 µs por lote
Tiempo optimizado :         5.69 µs por lote
Reducción del tiempo de CPU: 99.85 %
Aceleración: 674 veces más rápido


### 4.5 Costo del índice
El conjunto se construye **una sola vez**. Se mide su costo y su memoria para comprobar que la mejora no traslada la carga a otra parte del sistema.

In [ ]:
t_indice = %timeit -o -n 100 -r 7 set(historial_corriente_a)
print(f"Construcción del índice (una sola vez): {t_indice.average * 1e6:,.2f} µs")
print(f"Memoria de la lista del historial: {sys.getsizeof(historial_corriente_a):,} bytes")
print(f"Memoria adicional del índice: {sys.getsizeof(lecturas_registradas):,} bytes "
      f"({len(lecturas_registradas)} valores únicos)")

692 µs ± 40.6 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Construcción del índice (una sola vez): 691.79 µs
Memoria de la lista del historial: 240,056 bytes
Memoria adicional del índice: 728 bytes (6 valores únicos)
